In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy samplomatic
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
نوصي باستخدام هذه الإصدارات أو أحدث.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
samplomatic~=0.18.0
```
</AccordionItem>
</Accordion>

Executor primitive هو جزء من [نموذج التنفيذ الموجَّه](/guides/directed-execution-model)، الذي يوفر مرونة أكبر عند تخصيص سير عمل تخفيف الأخطاء.

تختلف مدخلات ومخرجات Executor primitive اختلافًا كبيرًا عن مدخلات ومخرجات [Sampler](/guides/sampler-input-output) و[Estimator](/guides/estimator-input-output) primitives. على سبيل المثال، بدلاً من أخذ قائمة من PUBs كمدخل، يأخذ Executor كائن `QuantumProgram`، الذي يحتوي على قائمة من كائنات `QuantumProgramItem`. تمنحك هذه الفئات الحاوية مرونة أكبر من PUB، الذي هو هيكل بيانات صف (tuple) بسيط.

مخرج Executor هو `QuantumProgramResult`، وهو قابل للتكرار ويحتوي على عنصر واحد لكل `QuantumProgramItem` مدخل.

<span id="programs"></span>

## المدخلات: البرامج الكمية

كما ذُكِر سابقًا، المدخل لـ Executor primitive هو [`QuantumProgram`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/quantum-program-quantum-program)، وهو قابل للتكرار من كائنات [`QuantumProgramItem`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/quantum-program-quantum-program-item). يمكن أن تكون هذه الكائنات من نوعين:
- `CircuitItem`، الذي يُخزِّن عادةً Circuit وقيم معاملاتها (إن وجدت).
- `SamplexItem`، الذي يُخزِّن عادةً ما يلي:
    - Circuit نموذجية
    - كائن samplex، يُستخدَم لتوليد مجموعات عشوائية من المعاملات في وقت التشغيل (مثلاً لإجراء التدوير أو حقن الضوضاء)
    - وسيطات للـ samplex، والتي قد تتضمن قيم معاملات للـ دائرة الأصلية

كل من هذه العناصر يُمثِّل مهمة مختلفة يجب على Executor تنفيذها.

### قبل البدء

بعض أمثلة الكود في هذه الصفحة تستخدم `samplex`، وهو جزء من حزمة Samplomatic. لذلك، قبل تشغيل تلك الكتل البرمجية، يجب تثبيت Samplomatic كما هو موضح في الكتلة البرمجية التالية. لمزيد من المعلومات، راجع [وثائق Samplomatic](https://qiskit.github.io/samplomatic).

In [1]:
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime.quantum_program import QuantumProgram
from qiskit_ibm_runtime import Executor, QiskitRuntimeService
from qiskit.circuit import Parameter, QuantumCircuit
import numpy as np
from samplomatic import build
from samplomatic.transpiler import generate_boxing_pass_manager

# Initialize an empty program
program = QuantumProgram(shots=1024)

# Initialize and transpile a 3-qubit quantum circuit with 2 parameters.
circuit = QuantumCircuit(3)
circuit.h(0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.rz(Parameter("theta"), 0)
circuit.rz(Parameter("phi"), 1)

# `measure_all` adds a 3-bit classical register named "meas"
circuit.measure_all()

# Choose the least busy backend
service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Generate a preset pass manager
# This will be used to convert the abstract circuit to an
# equivalent Instruction Set Architecture (ISA) circuit.
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Transpile the circuit
isa_circuit = preset_pass_manager.run(circuit)

### مثال: إنشاء `QuantumProgram` بمهمتين مختلفتين
أولاً هيِّئ برنامجك الكمي، ثم ألحِق عناصر البرنامج باستخدام `append_circuit_item` أو `append_samplex_item` (إذا كان samplex موجودًا)، كما هو موضح في الأمثلة التالية.

تُهيِّئ الخلية التالية `QuantumProgram` وتُحدِّد أنه يجب تشغيل 1024 تشغيلاً لكل تهيئة لكل عنصر في البرنامج.

> **Note:** على خلاف Sampler، يأخذ `QuantumProgram` قيمة تشغيلات واحدة فقط. إذا كنت تريد قيمة تشغيلات مختلفة، فأنت بحاجة إلى `QuantumProgram` منفصل، وهو سيكون مهمة منفصلة.

In [2]:
# Append the transpiled circuit and an array
# containing 10 sets of parameter values to the program
program.append_circuit_item(
    isa_circuit,
    circuit_arguments=np.random.rand(
        10, 2
    ),  # 10 sets of parameter values and 2 parameters
)

### إلحاق `CircuitItem`
بعد ذلك، ألحِق الـ دائرة المستهدفة التي جرى Transpile لها وفق مجموعة تعليمات معمارية الـ Backend (ISA) بـ `QuantumProgram`. بما أن هذه الـ دائرة تحتوي على معاملَين، يجب أيضًا توفير قيم المعاملات (10 مجموعات في هذا المثال). تنفيذ هذه `CircuitItem` هو المهمة الأولى التي سيؤديها البرنامج.

In [3]:
# Transpile the circuit, additionally grouping gates and measurements into annotated boxes
preset_pass_manager = generate_preset_pass_manager(
    backend=backend, optimization_level=0
)

# Use the boxing pass manager to group gates
# and measurements into boxes and add
# a`Twirl` annotation.
preset_pass_manager.post_scheduling = generate_boxing_pass_manager(
    # Add gate twirling
    enable_gates=True,
    # Add measurement twirling
    enable_measures=True,
)
boxed_circuit = preset_pass_manager.run(circuit)

# Build the template circuit and the samplex.  The template circuit has parametric gates
# without fixed values and the samplex randomly generates the parameter
# values on the server side at runtime to perform twirling.
template_circuit, samplex = build(boxed_circuit)

# Determine what arguments are required by the samplex.
# Input the arguments in samplex_arguments.
print(samplex.inputs())

TensorInterface(<
  - 'parameter_values' <float64[2]>: Input parameter values to use during sampling.
>)


In [4]:
# Append the template circuit and samplex as a samplex item
program.append_samplex_item(
    template_circuit,
    samplex=samplex,
    samplex_arguments={
        # the arguments required by the samplex.sample method
        "parameter_values": np.random.rand(10, 2),
    },
    shape=(28, 10),  # 28 randomizations and 10 sets of parameter values
)

In [5]:
# Initialize an Executor with the default options
executor = Executor(mode=backend)

# Submit the job
job = executor.run(program)

# Retrieve the result
result = job.result()

## Outputs

Executor's output is a [`QuantumProgramResult`](/docs/api/qiskit-ibm-runtime/results-quantum-program-result), which is an iterable. It contains one entry per input `QuantumProgramItem` in the same order as the input items. Each of these output items is a dictionary where the keys are strings that correspond the classical registers' names in the input circuits (among others), so you no longer need to memorize these names like you did with Sampler output. The dictionary values are of type `np.ndarray`.

The result for the previous example contains these items:

### `CircuitItem` result

The first item contains the results of running the first task (a `CircuitItem`) in the program. It contains a single key, `meas`, which is the name of the classical register in the input circuit. The value of this key maps to an `np.ndarray` of shape `(parameter sets, shots, register bits)`, which is (10, 1024, 3) for the above example.

The following code illustrates how to access this information:

In [6]:
# Access the results of the classical register of task #0, a CircuitItem
result_0 = result[0]["meas"]
print(f"Result shape: {result_0.shape}")

Result shape: (10, 1024, 3)


### `SamplexItem` result

The second item contains the results of running the second task (a `SamplexItem`) in the program. This item contains multiple keys. The `meas` key, which is the name of the input circuit's classical register, maps to that register's array of results. This array has the shape `(randomizations, parameter sets, shots, classical bits)`, or (28, 10, 1024, 3) in this example. Additionally, the output contains a `measurement_flips.meas` key, which is the bit-flip corrections to undo the measurement twirling for the `meas` register.  This output shape will be (28, 10, 1, 3) for our example because only one shot is required to perform the bit-flip.

In [7]:
# Access the results of the classical register of task #1
result_1 = result[1]["meas"]
print(f"Result shape: {result_1.shape}")

# Access the bit-flip corrections
flips_1 = result[1]["measurement_flips.meas"]
print(f"Bit-flip corrections shape: {flips_1.shape}")

# Undo the bit flips via classical XOR
unflipped_result_1 = result_1 ^ flips_1

Result shape: (28, 10, 1024, 3)
Bit-flip corrections shape: (28, 10, 1, 3)


## المخرجات
مخرج Executor هو [`QuantumProgramResult`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/results-quantum-program-result)، وهو قابل للتكرار. يحتوي على إدخال واحد لكل `QuantumProgramItem` مدخل بنفس ترتيب عناصر الإدخال. كل من هذه العناصر المخرجة هو قاموس حيث المفاتيح هي سلاسل نصية تتوافق مع أسماء السجلات الكلاسيكية في الـ Circuits المدخلة (من بين أمور أخرى)، بحيث لا تحتاج بعد الآن إلى حفظ هذه الأسماء كما كنت تفعل مع مخرج Sampler. قيم القاموس من النوع `np.ndarray`.

تحتوي نتيجة المثال السابق على هذه العناصر:

### نتيجة `CircuitItem`
يحتوي العنصر الأول على نتائج تشغيل المهمة الأولى (`CircuitItem`) في البرنامج. يحتوي على مفتاح واحد، `meas`، وهو اسم السجل الكلاسيكي في الـ دائرة المدخلة. قيمة هذا المفتاح تُعيَّن إلى `np.ndarray` بالشكل `(مجموعات المعاملات، التشغيلات، بتات السجل)`، وهو (10، 1024، 3) في المثال أعلاه.

الكود التالي يُوضِّح كيفية الوصول إلى هذه المعلومات: